## Structured concurrency

`CompletableFuture` is powerful but easy to misuse — orphaned tasks, leaked threads, error handling that quietly gets lost. **Structured concurrency** (preview in Java 21, finalising soon) treats a group of concurrent tasks as a single block: they share a lifetime, errors propagate up, and the block can't exit until every child is done or cancelled. The shape echoes try-with-resources.

```java
try (var scope = new StructuredTaskScope.ShutdownOnFailure()) {
    var userTask  = scope.fork(() -> fetchUser(id));
    var orderTask = scope.fork(() -> fetchOrders(id));

    scope.join();            // wait for both
    scope.throwIfFailed();   // if either threw, propagate

    return new Profile(userTask.get(), orderTask.get());
}
// on exit, any still-running fork is cancelled automatically
```

The structured model gives three guarantees the unstructured one didn't: tasks **never outlive** the block that started them, errors **never get swallowed**, and cancellation **propagates** automatically. Treat it as the future direction of concurrent Java — once it's out of preview, prefer it over hand-rolled `CompletableFuture` graphs.
